# Modulo 7 · Lezione 10 — Progetto Finale: California Housing

**Corso di Intelligenza Artificiale — Livello base**

---

## Obiettivo del progetto

Predire il **valore mediano delle abitazioni** nei blocchi della California usando i dati del censimento del 1990.
Applicheremo **tutto ciò che abbiamo imparato nel Modulo 7**:

| Fase | Contenuto |
|------|-----------|
| 1 | Caricare e ispezionare i dati |
| 2 | Analisi esplorativa (EDA) |
| 3 | Feature engineering |
| 4 | Pipeline con ColumnTransformer |
| 5 | Confrontare modelli multipli |
| 6 | Ottimizzare il migliore con GridSearchCV |
| 7 | Valutare e diagnosticare il modello finale |
| 8 | Salvare il modello con joblib |

---
```
Dataset: California Housing
  Fonte: sklearn.datasets.fetch_california_housing
  Campioni: 20640 blocchi censuari
  Feature: 8 (numeriche)
  Target: medianHouseValue (in centinaia di migliaia di $)
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import (
    train_test_split, cross_val_score, learning_curve,
    GridSearchCV, StratifiedKFold, KFold
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score
)

np.random.seed(42)
print("Librerie pronte")

---
## Fase 1 — Caricare e ispezionare i dati

In [ ]:
# Caricare il dataset
california = fetch_california_housing(as_frame=True)
df = california.frame.copy()

print("Descrizione del dataset:")
print(california.DESCR[:1500])

In [ ]:
print(f"Shape: {df.shape}")
print(f"\nFeature: {list(df.columns)}")
print(f"\nTipi di dato:")
print(df.dtypes)
print(f"\nValori nulli:")
print(df.isnull().sum())
print(f"\nStatistiche di base:")
print(df.describe().round(2))

---
## Fase 2 — Analisi esplorativa (EDA)

In [ ]:
target_col = 'MedHouseVal'

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribuzione del target
axes[0].hist(df[target_col], bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(df[target_col].mean(),   color='red', lw=2, ls='--', label=f'Media={df[target_col].mean():.2f}')
axes[0].axvline(df[target_col].median(), color='green', lw=2, ls=':', label=f'Mediana={df[target_col].median():.2f}')
axes[0].set_title('Distribuzione del prezzo (target)')
axes[0].set_xlabel('MedHouseVal (100k $)')
axes[0].legend(fontsize=8)

# Nota: il target è troncato a 5.0 (500k $)
print(f"Percentuale di campioni al limite massimo (5.0): "
      f"{(df[target_col] == 5.0).mean():.2%}")

# Distribuzione geografica
scatter = axes[1].scatter(df['Longitude'], df['Latitude'],
                          c=df[target_col], cmap='viridis', s=2, alpha=0.5)
plt.colorbar(scatter, ax=axes[1], label='MedHouseVal')
axes[1].set_title('Distribuzione geografica del prezzo')
axes[1].set_xlabel('Longitudine'); axes[1].set_ylabel('Latitudine')

# Correlazioni con il target
corr = df.corr()[target_col].drop(target_col).sort_values()
corr.plot(kind='barh', ax=axes[2], color=['coral' if v < 0 else 'steelblue' for v in corr])
axes[2].set_title(f'Correlazione con {target_col}')
axes[2].axvline(0, color='black', lw=0.8)

plt.tight_layout(); plt.show()

In [ ]:
# Distribuzione di tutte le feature numeriche
features_num = [c for c in df.columns if c != target_col]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.ravel(), features_num):
    ax.hist(df[col], bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(col)
    ax.set_xlabel('')
    q99 = df[col].quantile(0.99)
    ax.axvline(q99, color='red', ls='--', lw=1.5, label=f'P99={q99:.1f}')
    ax.legend(fontsize=7)

plt.suptitle('Distribuzione delle feature numeriche', fontsize=12)
plt.tight_layout(); plt.show()

print("Osservazioni:")
for col in features_num:
    skew = df[col].skew()
    print(f"  {col:20}: skewness={skew:.2f}{'  ← molto asimmetrica' if abs(skew)>2 else ''}")

---
## Fase 3 — Feature engineering

In [ ]:
# Nuove feature rilevanti
df['habitaciones_por_casa'] = df['AveRooms']   / df['HouseAge'].clip(1)
df['dormitorios_por_hab']   = df['AveBedrms']  / df['AveRooms'].clip(0.1)
df['densidad_poblacion']    = df['Population'] / df['AveOccup'].clip(1)
df['ingreso_por_persona']   = df['MedInc']     / df['AveOccup'].clip(1)

# Nuove correlazioni con il target
nuevas_features = ['habitaciones_por_casa','dormitorios_por_hab',
                   'densidad_poblacion','ingreso_por_persona']

corr_nuevas = df[nuevas_features + [target_col]].corr()[target_col].drop(target_col)
corr_originales = df[features_num + [target_col]].corr()[target_col].drop(target_col)

print("Correlazioni originali con il target:")
print(corr_originales.round(3).to_string())
print("\nNuove feature:")
print(corr_nuevas.round(3).to_string())

In [ ]:
# Preparare X e y
features_finales = features_num + nuevas_features
X = df[features_finales]
y = df[target_col]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train: {X_tr.shape[0]} campioni")
print(f"Test:  {X_te.shape[0]} campioni")
print(f"Feature: {len(features_finales)}")

---
## Fase 4 — Pipeline con preprocessing

In [ ]:
# Tutte le feature sono numeriche in questo dataset
preprocesador = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

def crear_pipeline_regresion(modelo):
    return Pipeline([
        ('prepro', preprocesador),
        ('modelo', modelo),
    ])

def evaluar_regresion(y_real, y_pred, nombre=''):
    mae  = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    r2   = r2_score(y_real, y_pred)
    return {'Modelo': nombre, 'MAE': mae, 'RMSE': rmse, 'R²': r2}

print("Pipeline definita")

---
## Fase 5 — Confrontare modelli multipli

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

modelos = {
    'Baseline (media)':        None,  # modello banale
    'LinearRegression':        LinearRegression(),
    'Ridge α=1':               Ridge(alpha=1),
    'Lasso α=0.01':            Lasso(alpha=0.01),
    'RandomForest 100':        RandomForestRegressor(100, random_state=42, n_jobs=-1),
    'GradientBoosting 100':    GradientBoostingRegressor(100, random_state=42),
}

resultados = []

for nombre, m in modelos.items():
    if m is None:
        # Baseline: predice sempre la media del train
        y_pred_base = np.full(len(y_te), y_tr.mean())
        resultados.append(evaluar_regresion(y_te, y_pred_base, nombre))
        continue

    pipe = crear_pipeline_regresion(m)
    pipe.fit(X_tr, y_tr)
    y_pred = pipe.predict(X_te)
    resultados.append(evaluar_regresion(y_te, y_pred, nombre))

df_res = pd.DataFrame(resultados).sort_values('R²', ascending=False)
print("Confronto tra modelli sul test:")
print(df_res.round(4).to_string(index=False))

# Grafico
fig, ax = plt.subplots(figsize=(10, 4))
colores = ['coral' if r < 0.5 else 'steelblue' if r < 0.8 else 'seagreen'
           for r in df_res['R²']]
bars = ax.bar(df_res['Modelo'], df_res['R²'], color=colores)
ax.set_title('R² sul test per modello')
ax.set_ylabel('R²')
ax.tick_params(axis='x', rotation=25)
for bar, r2 in zip(bars, df_res['R²']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{r2:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout(); plt.show()

---
## Fase 6 — Ottimizzare il modello migliore con GridSearchCV

In [ ]:
# GradientBoosting di solito vince nei dataset tabulari
pipe_gb = crear_pipeline_regresion(
    GradientBoostingRegressor(random_state=42)
)

rejilla = {
    'modelo__n_estimators':   [100, 200],
    'modelo__max_depth':      [3, 4, 5],
    'modelo__learning_rate':  [0.05, 0.1, 0.2],
    'modelo__subsample':      [0.8, 1.0],
}

gs = GridSearchCV(
    pipe_gb, rejilla,
    cv=KFold(5, shuffle=True, random_state=42),
    scoring='r2',
    refit=True, n_jobs=-1, verbose=1
)
gs.fit(X_tr, y_tr)

print(f"\nMigliori parametri: {gs.best_params_}")
print(f"R² CV (train): {gs.best_score_:.4f}")

---
## Fase 7 — Valutare e diagnosticare il modello finale

In [ ]:
modelo_final = gs.best_estimator_
y_pred_final = modelo_final.predict(X_te)

metricas = evaluar_regresion(y_te, y_pred_final, 'GradientBoosting ottimizzato')
print("Metriche sul test — modello finale:")
for k, v in metricas.items():
    if k != 'Modelo':
        print(f"  {k:6}: {v:.4f}")
print(f"\nInterpretazione:")
print(f"  MAE  = {metricas['MAE']:.4f} → errore medio di {metricas['MAE']*100_000:.0f} $")
print(f"  RMSE = {metricas['RMSE']:.4f} → penalizza gli errori grandi")
print(f"  R²   = {metricas['R²']:.4f} → il modello spiega il {metricas['R²']*100:.1f}% della varianza")

In [ ]:
residuos = y_te - y_pred_final

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# 1. Reale vs Predetto
axes[0,0].scatter(y_te, y_pred_final, s=5, alpha=0.3, color='steelblue')
rango = [min(y_te.min(), y_pred_final.min()), max(y_te.max(), y_pred_final.max())]
axes[0,0].plot(rango, rango, 'r--', lw=1.5, label='Predizione perfetta')
axes[0,0].set_xlabel('Valore reale'); axes[0,0].set_ylabel('Valore predetto')
axes[0,0].set_title(f'Reale vs Predetto (R²={metricas["R²"]:.4f})')
axes[0,0].legend(fontsize=8)

# 2. Residui vs Predetto
axes[0,1].scatter(y_pred_final, residuos, s=5, alpha=0.3, color='coral')
axes[0,1].axhline(0, color='black', lw=1.5, ls='--')
axes[0,1].set_xlabel('Valore predetto'); axes[0,1].set_ylabel('Residui')
axes[0,1].set_title('Residui vs Predetto')

# 3. Istogramma dei residui
axes[1,0].hist(residuos, bins=50, color='seagreen', edgecolor='white', alpha=0.8)
axes[1,0].axvline(0, color='red', lw=2, ls='--')
axes[1,0].set_xlabel('Residui')
axes[1,0].set_title(f'Distribuzione residui (media={residuos.mean():.4f}, std={residuos.std():.4f})')

# 4. Feature importance
gb = modelo_final.named_steps['modelo']
importancias = pd.Series(gb.feature_importances_, index=features_finales).sort_values(ascending=True)
importancias.plot(kind='barh', ax=axes[1,1], color='mediumpurple')
axes[1,1].set_title('Feature Importances')
axes[1,1].set_xlabel('Importanza')

plt.suptitle('Diagnosi del modello finale — GradientBoosting ottimizzato', fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# Curva di apprendimento del modello finale
sizes, tr_scores, te_scores = learning_curve(
    modelo_final, X_tr, y_tr,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=5, scoring='r2', n_jobs=-1
)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(sizes, tr_scores.mean(1), 'b-o', ms=6, lw=2, label='Train R²')
ax.plot(sizes, te_scores.mean(1), 'r-o', ms=6, lw=2, label='Val CV R²')
ax.fill_between(sizes, tr_scores.mean(1)-tr_scores.std(1),
                tr_scores.mean(1)+tr_scores.std(1), alpha=0.1, color='blue')
ax.fill_between(sizes, te_scores.mean(1)-te_scores.std(1),
                te_scores.mean(1)+te_scores.std(1), alpha=0.1, color='red')
ax.set_xlabel('N. campioni di training')
ax.set_ylabel('R²')
ax.set_title('Curva di apprendimento — GradientBoosting ottimizzato')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

gap = tr_scores.mean(1)[-1] - te_scores.mean(1)[-1]
print(f"Gap finale (train - val): {gap:.4f}")
print(f"Diagnosi: {'BUON ADATTAMENTO' if gap < 0.08 else 'LIEVE OVERFITTING'}")

---
## Fase 8 — Salvare il modello

In [ ]:
# Salvare la pipeline completa
ruta = 'california_housing_model_v1.joblib'
joblib.dump(modelo_final, ruta)

# Metadati
meta = {
    'version':        'v1.0',
    'dataset':        'California Housing (sklearn)',
    'features':       features_finales,
    'target':         'MedHouseVal (100k $)',
    'algoritmo':      'GradientBoostingRegressor',
    'mejores_params': {k: str(v) for k, v in gs.best_params_.items()},
    'r2_cv_train':    round(gs.best_score_, 4),
    'mae_test':       round(metricas['MAE'], 4),
    'rmse_test':      round(metricas['RMSE'], 4),
    'r2_test':        round(metricas['R²'], 4),
    'n_train':        len(X_tr),
    'n_test':         len(X_te),
}
with open('california_housing_model_v1_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print("Modello salvato:")
print(f"  {ruta}")
print(f"  california_housing_model_v1_meta.json")

# Verificare che il modello caricato funzioni
modelo_cargado = joblib.load(ruta)
y_pred_check   = modelo_cargado.predict(X_te[:3])
print(f"\nVerifica (3 predizioni):")
print(f"  Predetto (100k $): {y_pred_check.round(3)}")
print(f"  Reale    (100k $): {y_te.values[:3].round(3)}")

---
## Report finale del progetto

```
═══════════════════════════════════════════════════════════
  PROGETTO: Predizione dei prezzi delle case in California
═══════════════════════════════════════════════════════════

DATASET:
  20640 blocchi censuari, 8 feature originali + 4 derivate
  Target: valore mediano dell'abitazione (100k $)

PIPELINE:
  → SimpleImputer (mediana) → StandardScaler → GradientBoosting

CONFRONTO TRA MODELLI:
  Baseline (media)       R²≈0.00 (riferimento banale)
  LinearRegression       R²≈0.60
  Ridge                  R²≈0.60
  RandomForest           R²≈0.80
  GradientBoosting opt   R²≈0.83

MODELLO FINALE:
  GradientBoostingRegressor con iperparametri ottimizzati
  MAE  ≈ 0.31 (31k $ di errore medio)
  RMSE ≈ 0.45
  R²   ≈ 0.83 (spiega l'83% della varianza)

FEATURE PIÙ IMPORTANTI:
  1. MedInc              (reddito mediano del quartiere)
  2. Latitude / Longitude (localizzazione geografica)
  3. ingreso_por_persona  (feature derivata)

DIAGNOSI:
  Curva di apprendimento: buon adattamento, senza overfitting significativo
  Residui: distribuzione simmetrica centrata su 0
  Limite del cap a 5.0 → introduce rumore nelle predizioni alte
═══════════════════════════════════════════════════════════
```

---
## Pratica

### Esercizio 1 — Analizzare l'impatto delle feature derivate

**Consegna:**

1. Allena due modelli identici (GradientBoosting con parametri di default):
   - Modello A: solo feature originali (8)
   - Modello B: feature originali + derivate (12)
2. Confronta R², MAE e RMSE sul test
3. Quanto migliorano le feature derivate?
4. Quale feature derivata ha più impatto? (feature importance)

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

california = fetch_california_housing(as_frame=True)
df = california.frame.copy()

# Feature derivate
df['habitaciones_por_casa'] = df['AveRooms']   / df['HouseAge'].clip(1)
df['dormitorios_por_hab']   = df['AveBedrms']  / df['AveRooms'].clip(0.1)
df['densidad_poblacion']    = df['Population'] / df['AveOccup'].clip(1)
df['ingreso_por_persona']   = df['MedInc']     / df['AveOccup'].clip(1)

target_col = 'MedHouseVal'
features_orig = [c for c in california.frame.columns if c != target_col]
features_full = features_orig + ['habitaciones_por_casa','dormitorios_por_hab',
                                  'densidad_poblacion','ingreso_por_persona']
y = df[target_col]

resultados = []
for nombre, feats in [('Solo originali', features_orig), ('+ Derivate', features_full)]:
    X_ = df[feats]
    X_tr_, X_te_, y_tr_, y_te_ = train_test_split(X_, y, test_size=0.2, random_state=42)
    pipe = Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc',  StandardScaler()),
        ('m',   GradientBoostingRegressor(100, random_state=42)),
    ])
    pipe.fit(X_tr_, y_tr_)
    y_pred_ = pipe.predict(X_te_)
    r2  = r2_score(y_te_, y_pred_)
    mae = mean_absolute_error(y_te_, y_pred_)
    rmse = np.sqrt(mean_squared_error(y_te_, y_pred_))
    resultados.append({'Modelo': nombre, 'R²': r2, 'MAE': mae, 'RMSE': rmse})
    # Feature importance
    imp = pd.Series(pipe['m'].feature_importances_, index=feats).sort_values(ascending=False)
    print(f"\n{nombre}: R²={r2:.4f}, MAE={mae:.4f}")
    print(f"Top 5 feature: {imp.head(5).index.tolist()}")

df_comp = pd.DataFrame(resultados)
print(f"\nMiglioramento dovuto alle feature derivate:")
mejora_r2 = df_comp.iloc[1]['R²'] - df_comp.iloc[0]['R²']
print(f"  R²: +{mejora_r2:.4f}")
```
</details>

---
### Esercizio 2 — Analisi degli errori per zona geografica

**Consegna:**

1. Usa il modello finale del progetto
2. Aggiungi al DataFrame di test: `y_real`, `y_pred`, `error_abs` (|reale - predetto|)
3. Visualizza in una mappa scatter (Latitude, Longitude) colorata per l'errore assoluto
4. In quali zone geografiche il modello commette più errori?
5. Cosa potresti fare per migliorare in quelle zone?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Ricostruire X_te con le coordinate geografiche
X_te_full = df.loc[X_te.index, features_finales + ['Latitude','Longitude']]

df_errores = X_te_full[['Latitude','Longitude']].copy()
df_errores['y_real'] = y_te.values
df_errores['y_pred'] = y_pred_final
df_errores['error_abs'] = np.abs(y_te.values - y_pred_final)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Mappa degli errori
sc1 = axes[0].scatter(
    df_errores['Longitude'], df_errores['Latitude'],
    c=df_errores['error_abs'], cmap='YlOrRd', s=4, alpha=0.6
)
plt.colorbar(sc1, ax=axes[0], label='Errore assoluto (100k $)')
axes[0].set_title('Errore assoluto per zona geografica')
axes[0].set_xlabel('Longitudine'); axes[0].set_ylabel('Latitudine')

# Top 10% errori
umbral = df_errores['error_abs'].quantile(0.9)
peores = df_errores[df_errores['error_abs'] >= umbral]
sc2 = axes[1].scatter(
    df_errores['Longitude'], df_errores['Latitude'],
    c='lightgray', s=3, alpha=0.3
)
axes[1].scatter(
    peores['Longitude'], peores['Latitude'],
    c=peores['error_abs'], cmap='Reds', s=15, alpha=0.8
)
axes[1].set_title('Top 10% errori più grandi')
axes[1].set_xlabel('Longitudine'); axes[1].set_ylabel('Latitudine')

plt.tight_layout(); plt.show()

# Analisi per zona
print("Statistiche per decile di prezzo reale:")
df_errores['decil'] = pd.qcut(df_errores['y_real'], 10, labels=False)
print(df_errores.groupby('decil')['error_abs'].mean().round(4).to_string())
print("\nOsservazione: gli errori sono maggiori nelle case costose (vicine al cap di 5.0).")
print("Miglioramento possibile: filtrare o separare le case al limite massimo.")
```
</details>

---
### Esercizio 3 — Provare RandomizedSearchCV per GradientBoosting

**Consegna:** Il GridSearchCV ha esplorato 36 combinazioni. Verifica se con `RandomizedSearchCV` (n_iter=20) ottieni risultati simili in meno tempo:

1. Definisci uno spazio di ricerca più ampio con distribuzioni continue
2. Esegui `RandomizedSearchCV` con n_iter=20, cv=5
3. Confronta tempo e R² con il GridSearchCV precedente
4. Quale consiglieresti in produzione con iperparametri continui?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
import time
from scipy.stats import randint, uniform
from sklearn.model_selection import RandomizedSearchCV, KFold

espacio_rs = {
    'modelo__n_estimators':  randint(50, 300),
    'modelo__max_depth':     randint(2, 7),
    'modelo__learning_rate': uniform(0.01, 0.25),
    'modelo__subsample':     uniform(0.6, 0.4),
    'modelo__min_samples_split': randint(2, 10),
}

pipe_rs = crear_pipeline_regresion(GradientBoostingRegressor(random_state=42))
rs = RandomizedSearchCV(
    pipe_rs, espacio_rs, n_iter=20,
    cv=KFold(5, shuffle=True, random_state=42),
    scoring='r2', refit=True, n_jobs=-1, random_state=42
)

t0 = time.time()
rs.fit(X_tr, y_tr)
t_rs = time.time() - t0

y_pred_rs = rs.predict(X_te)
r2_rs = r2_score(y_te, y_pred_rs)

print(f"RandomizedSearchCV (n_iter=20): tempo={t_rs:.1f}s, R²={r2_rs:.4f}")
print(f"GridSearchCV (36 combinazioni): R²={metricas['R²']:.4f}")
print(f"\nMigliori parametri RS: {rs.best_params_}")
print("\nVerdetto: RandomSearch con uno spazio più ampio di solito eguaglia o supera")
print("GridSearch con uno spazio discreto, usando meno valutazioni.")
```
</details>

---
### Esercizio 4 — Classificazione alternativa del prezzo

**Consegna:** Trasforma il problema in classificazione:

```python
# 3 categorie di prezzo
y_cat = pd.cut(y, bins=[0, 1.5, 3.0, 6.0], labels=['basso','medio','alto'])
```

1. Allena una pipeline di classificazione (GradientBoosting o RF) con le stesse feature
2. Valuta con 5-Fold CV: accuracy e F1 macro
3. Mostra la matrice di confusione sul test
4. Quale categoria confonde di più il modello?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import numpy as np

y_cat = pd.cut(y, bins=[0, 1.5, 3.0, 6.0], labels=['basso','medio','alto'])
X_clf = X.copy()
# Rimuovere i NaN dal target
mask = y_cat.notna()
X_clf, y_clf = X_clf[mask], y_cat[mask]

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X_clf, y_clf, test_size=0.2, stratify=y_clf, random_state=42
)

pipe_clf = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('sc',  StandardScaler()),
    ('clf', GradientBoostingClassifier(100, random_state=42)),
])

cv = StratifiedKFold(5, shuffle=True, random_state=42)
acc = cross_val_score(pipe_clf, X_clf, y_clf, cv=cv, scoring='accuracy', n_jobs=-1).mean()
f1  = cross_val_score(pipe_clf, X_clf, y_clf, cv=cv, scoring='f1_macro', n_jobs=-1).mean()
print(f"Accuracy CV: {acc:.4f}")
print(f"F1 macro CV: {f1:.4f}")

pipe_clf.fit(X_tr_c, y_tr_c)
print(f"\n{classification_report(y_te_c, pipe_clf.predict(X_te_c))}")

fig, ax = plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay.from_estimator(pipe_clf, X_te_c, y_te_c, ax=ax, cmap='Blues')
ax.set_title('Matrice di confusione — Classificazione del prezzo')
plt.tight_layout(); plt.show()
print("La categoria 'medio' tende ad avere più confusione con 'basso' e 'alto'.")
```
</details>

---
### Esercizio 5 (sfida) — Il tuo dataset, il tuo progetto

**Consegna:** Applica l'intero processo del progetto a un dataset diverso di tua scelta:

```python
# Opzioni
from sklearn.datasets import load_diabetes      # regressione: progressione del diabete
from sklearn.datasets import load_breast_cancer # classificazione: diagnosi del cancro
from sklearn.datasets import load_wine          # classificazione: tipo di vino
# Oppure scarica un dataset di Kaggle che ti interessa
```

Il tuo progetto deve includere:
1. EDA con almeno 3 visualizzazioni
2. Almeno 1 feature derivata
3. Pipeline con ColumnTransformer (anche se tutte le feature sono numeriche)
4. Confronto di almeno 3 modelli
5. GridSearchCV o RandomizedSearchCV
6. Diagnosi (curva di apprendimento + residui o matrice di confusione)
7. Salvare il modello con joblib
8. Report finale: 5 conclusioni sul dataset e sul modello

In [ ]:
# ✏️ IL TUO CODICE QUI
# Questo è il tuo progetto libero. Non esiste un'unica soluzione!
# Applica il processo delle 8 fasi di questo notebook a un dataset di tua scelta.


---
## Riepilogo del Modulo 7 — Machine Learning

```python
# ═══════════════════════════════════════════════════════════
# RIEPILOGO COMPLETO DEL MODULO 7
# ═══════════════════════════════════════════════════════════

# C1: API universale di sklearn
modelo.fit(X_tr, y_tr)
modelo.predict(X_te)
modelo.score(X_te, y_te)

# C2: Preparazione dei dati (senza data leakage)
ct = ColumnTransformer([
    ('num', StandardScaler(), cols_num),
    ('nom', OneHotEncoder(drop='first'), cols_nom),
])
X_tr_t = ct.fit_transform(X_tr)
X_te_t = ct.transform(X_te)          # NO fit sul test

# C3: Regressione
mae  = mean_absolute_error(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))
r2   = r2_score(y, y_pred)

# C4: Classificazione
acc  = accuracy_score(y, y_pred)
f1   = f1_score(y, y_pred)
auc  = roc_auc_score(y, proba)

# C5: Alberi e Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
importancias = rf.feature_importances_

# C6: Overfitting → curve di diagnosi
sizes, tr_sc, te_sc = learning_curve(modelo, X, y, cv=5)
gap = tr_sc.mean(1)[-1] - te_sc.mean(1)[-1]  # grande → overfitting

# C7: Cross-Validation e GridSearch
scores = cross_val_score(modelo, X, y, cv=5, scoring='accuracy')
gs = GridSearchCV(modelo, param_grid, cv=5, refit=True)

# C8: Clustering
X_sc = StandardScaler().fit_transform(X)
km   = KMeans(n_clusters=4, n_init=20, random_state=42)
sc   = silhouette_score(X_sc, km.fit_predict(X_sc))

# C9: Pipeline completa
pipe = Pipeline([('prepro', ct), ('clf', RandomForestClassifier())])
gs   = GridSearchCV(pipe, {'clf__n_estimators': [50,100]}, cv=5)
joblib.dump(gs.best_estimator_, 'modelo.joblib')
modelo = joblib.load('modelo.joblib')

# C10: Progetto end-to-end
#  EDA → Feature Eng → Pipeline → Confronto → GridSearch → Diagnosi → Salvataggio
```

---

## Congratulazioni!

Hai completato il **Modulo 7 — Machine Learning**.

**Il prossimo modulo** entrerà nel mondo del **Deep Learning**: reti neurali, PyTorch e le basi dell'apprendimento profondo che alimenta l'IA moderna.